In [1]:
# 02_logit_woe_vs_stump.ipynb
# Coefficient-level isomorphism: logistic-WoE beta  vs  boosted-stump cumulative leaf curve s(x).
# Two regimes:
#   (I)  IDEAL  — stumps constrained to the WoE bin grid  -> s(b) should recover WoE(b), beta_eff ~ slope.
#   (II) REALITY — stumps free on raw features           -> measure where splits & s(x) diverge from WoE.
# Single-cell convention; project-root-relative paths; figures in seaborn greyscale, dpi 600, png+pdf.

import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr, pearsonr
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
import statsmodels.api as sm
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
sns.set_theme(style='whitegrid', context='paper')
sns.set_palette('Greys')
plt.rcParams.update({'image.cmap': 'Greys', 'axes.edgecolor': '0.2',
                     'axes.grid': True, 'grid.color': '0.85', 'font.size': 10})

ROOT   = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA   = os.path.join(ROOT, 'data')
TABLES = os.path.join(ROOT, 'results', 'tables')
FIGS   = os.path.join(ROOT, 'results', 'figures')

def savefig(fig, name):
    """Save greyscale figure as png + pdf at dpi 600, no caption."""
    for ext in ('png', 'pdf'):
        fig.savefig(os.path.join(FIGS, name + '.' + ext), dpi=600, bbox_inches='tight')
    plt.close(fig)

# ===================================================================
# 0. LOAD shared WoE assets from 01
# ===================================================================
woe_X = pd.read_parquet(os.path.join(DATA, 'gmsc_woe.parquet'))
dfc   = pd.read_parquet(os.path.join(DATA, 'gmsc_clean.parquet'))
meta  = json.load(open(os.path.join(DATA, 'woe_maps.json')))
TARGET   = meta['target']; FEATURES = meta['features']; MAPS = meta['maps']
y = woe_X[TARGET].values.astype(int)
Xw = woe_X[FEATURES].copy()                     # WoE-transformed design
print('WoE matrix:', Xw.shape, '| bad rate %.4f' % y.mean())

# Per-feature ordered bin table (finite bins only, low->high) and a raw->bin mapping.
bin_order, bin_woe, bin_w = {}, {}, {}
for c in FEATURES:
    recs = [r for r in MAPS[c] if r['bin'] >= 0]
    recs = sorted(recs, key=lambda r: r['lo'])
    miss = [r for r in MAPS[c] if r['bin'] == -1]
    allrecs = recs + miss                        # missing appended last
    bin_order[c] = allrecs
    bin_woe[c]   = np.array([r['woe'] for r in allrecs])
    bin_w[c]     = np.array([r['n']   for r in allrecs], float)

def to_bin_index(x, recs):
    """Map raw values to ordinal bin position within recs (missing -> last)."""
    idx = np.full(len(x), len(recs) - 1, int)     # default = missing slot if present
    fin = [r for r in recs if r['bin'] >= 0]
    for pos, r in enumerate(fin):
        sel = (~x.isna()).values & (x > r['lo']).values & (x <= r['hi']).values
        idx[sel] = pos
    has_missing = any(r['bin'] == -1 for r in recs)
    if has_missing:
        idx[x.isna().values] = len(recs) - 1
    else:
        idx[x.isna().values] = 0
    return idx

Xbin = pd.DataFrame({c: to_bin_index(dfc[c], bin_order[c]) for c in FEATURES})

# ===================================================================
# 1. LOGISTIC-WoE  (statsmodels for coefficient SE / Wald CI)
# ===================================================================
Xw_sm = sm.add_constant(Xw)
logit = sm.Logit(y, Xw_sm).fit(disp=0)
beta      = logit.params.drop('const')
beta_se   = logit.bse.drop('const')
beta_ci   = logit.conf_int().drop('const')
beta_ci.columns = ['ci_lo', 'ci_hi']
print('\n=== Logistic-WoE coefficients ===')
logit_tab = pd.concat([beta.rename('beta'), beta_se.rename('se'), beta_ci], axis=1)
logit_tab.index.name = 'feature'
print(logit_tab.round(4).to_string())
print('Pseudo R2 = %.4f' % logit.prsquared)

# ===================================================================
# 2. REGIME I — IDEAL: univariate boosted stumps on the WoE bin grid.
#    For each feature we boost depth-1 trees on its (single) bin-index column.
#    Because a depth-1 tree on one ordinal column can only cut between bins,
#    every leaf is a union of bins; with enough rounds the cumulative leaf
#    function s(.) saturates to the per-bin log-odds deviation = WoE(bin).
#    F0 (base) = overall log-odds; we recover s per bin and regress on WoE.
# ===================================================================
F0 = np.log(y.mean() / (1 - y.mean()))

def univariate_stump_curve(bin_col, n_bins, y, lr=0.1, n_est=400):
    """Fit univariate stump-boosting on one ordinal bin column; return s per bin."""
    Xcol = bin_col.reshape(-1, 1).astype(float)
    gb = GradientBoostingClassifier(max_depth=1, learning_rate=lr,
                                    n_estimators=n_est, subsample=1.0,
                                    random_state=SEED)
    gb.fit(Xcol, y)
    # raw additive score (log-odds scale) at each bin level, minus base F0
    grid = np.arange(n_bins).reshape(-1, 1).astype(float)
    score = gb.decision_function(grid)            # = F0 + sum nu*delta over rounds
    s = score - F0
    return s, gb

def beta_eff(s, woe, w):
    """Population-weighted through-origin slope of s on WoE (doc eq.)."""
    num = np.sum(w * s * woe); den = np.sum(w * woe**2)
    return num / den if den > 0 else np.nan

iso_rows, s_curves = [], {}
for c in FEATURES:
    nb = len(bin_order[c])
    s, _ = univariate_stump_curve(Xbin[c].values, nb, y)
    woe = bin_woe[c]; w = bin_w[c]
    s_curves[c] = s
    be = beta_eff(s, woe, w)
    # weighted correlation of s vs woe
    wm_s = np.sum(w*s)/w.sum(); wm_w = np.sum(w*woe)/w.sum()
    cov = np.sum(w*(s-wm_s)*(woe-wm_w)); vs = np.sum(w*(s-wm_s)**2); vw = np.sum(w*(woe-wm_w)**2)
    wcorr = cov/np.sqrt(vs*vw) if vs>0 and vw>0 else np.nan
    nz = np.abs(woe) > 1e-9
    sign_match = float(np.mean(np.sign(s[nz]) == np.sign(woe[nz]))) if nz.any() else np.nan
    rank = spearmanr(s, woe).correlation if nb > 2 else np.nan
    iso_rows.append([c, nb, be, wcorr, sign_match, rank,
                     float(np.max(np.abs(s - be*woe)))])
iso_df = pd.DataFrame(iso_rows, columns=['feature','n_bins','beta_eff','w_corr_s_woe',
                                         'sign_match','spearman_s_woe','max_abs_resid'])
print('\n=== REGIME I (ideal): stump cumulative-leaf curve vs WoE ===')
print(iso_df.round(4).to_string(index=False))

# ===================================================================
# 3. REGIME II — REALITY: full multivariate stump-boosting on RAW features.
#    Recover each feature's partial score s_c(x) = sum over that feature's
#    stumps of nu*delta, then collapse to per-bin means to compare with WoE.
# ===================================================================
raw = dfc[FEATURES].copy()
raw_filled = raw.fillna(raw.median(numeric_only=True))   # trees need numeric; NaN->median
gb_full = GradientBoostingClassifier(max_depth=1, learning_rate=0.1,
                                     n_estimators=600, subsample=1.0, random_state=SEED)
gb_full.fit(raw_filled.values, y)

# Which feature each stump split on, and accumulate its contribution per row.
lr = gb_full.learning_rate
partial = {c: np.zeros(len(raw)) for c in FEATURES}
for stage in gb_full.estimators_:
    tree = stage[0].tree_
    fidx = tree.feature[0]                     # root split feature (depth-1 -> single split)
    if fidx < 0:
        continue
    fname = FEATURES[fidx]
    leaf = stage[0].apply(raw_filled.values)   # leaf id per row
    val  = tree.value.ravel()                  # leaf raw values
    partial[fname] += lr * val[leaf]

real_rows = []
s_real_bin = {}
for c in FEATURES:
    # average partial score within each WoE bin -> comparable to WoE(bin)
    idx = Xbin[c].values; nb = len(bin_order[c])
    s_bin = np.array([partial[c][idx==b].mean() if (idx==b).any() else 0.0 for b in range(nb)])
    s_real_bin[c] = s_bin
    woe = bin_woe[c]; w = bin_w[c]
    be = beta_eff(s_bin, woe, w)
    wm_s = np.sum(w*s_bin)/w.sum(); wm_w = np.sum(w*woe)/w.sum()
    cov = np.sum(w*(s_bin-wm_s)*(woe-wm_w)); vs=np.sum(w*(s_bin-wm_s)**2); vw=np.sum(w*(woe-wm_w)**2)
    wcorr = cov/np.sqrt(vs*vw) if vs>0 and vw>0 else np.nan
    used = int(sum(1 for st in gb_full.estimators_
                   if st[0].tree_.feature[0]==FEATURES.index(c)))
    real_rows.append([c, be, wcorr, used])
real_df = pd.DataFrame(real_rows, columns=['feature','beta_eff_raw','w_corr_raw','n_stumps_used'])
print('\n=== REGIME II (reality): raw-feature stump partial score vs WoE ===')
print(real_df.round(4).to_string(index=False))

# ===================================================================
# 4. SAVE TABLES
# ===================================================================
logit_tab.round(6).to_csv(os.path.join(TABLES, '02_logit_woe_coeffs.csv'))
iso_df.round(6).to_csv(os.path.join(TABLES, '02_isomorphism_ideal.csv'), index=False)
real_df.round(6).to_csv(os.path.join(TABLES, '02_isomorphism_reality.csv'), index=False)

# ===================================================================
# 5. FIGURE A — s(b) vs WoE(b) scatter, ideal vs reality (greyscale)
# ===================================================================
alls_ideal = np.concatenate([s_curves[c] for c in FEATURES])
allw       = np.concatenate([bin_woe[c] for c in FEATURES])
alls_real  = np.concatenate([s_real_bin[c] for c in FEATURES])
lim = np.array([min(allw.min(), alls_ideal.min(), alls_real.min()),
                max(allw.max(), alls_ideal.max(), alls_real.max())]) * 1.05
fig, axes = plt.subplots(1, 2, figsize=(10, 4.6))
for ax, sv, ttl in [(axes[0], alls_ideal, 'Regime I: stumps on WoE grid'),
                    (axes[1], alls_real, 'Regime II: stumps on raw features')]:
    ax.plot(lim, lim, color='0.0', lw=1, ls='--', zorder=1)
    ax.scatter(allw, sv, s=26, c='0.35', edgecolor='0.0', linewidth=.4, zorder=2)
    r = pearsonr(allw, sv)[0]
    ax.set_title('%s  (r=%.3f)' % (ttl, r))
    ax.set_xlabel('WoE(bin)'); ax.set_ylabel('cumulative stump score s(bin)')
    ax.set_xlim(lim); ax.set_ylim(lim); ax.set_aspect('equal')
savefig(fig, '02_s_vs_woe_scatter')

# ===================================================================
# 6. FIGURE B — beta (logit) vs beta_eff (stump slope), ideal & reality
# ===================================================================
comp = logit_tab[['beta']].join(iso_df.set_index('feature')['beta_eff']) \
                          .join(real_df.set_index('feature')['beta_eff_raw'])
fig, ax = plt.subplots(figsize=(6.2, 5.0))
lo = min(comp.min().min(), 0); hi = comp.max().max()*1.08
ax.plot([lo,hi],[lo,hi], color='0.0', lw=1, ls='--', zorder=1)
ax.scatter(comp['beta'], comp['beta_eff'], s=70, marker='o', c='0.3',
           edgecolor='0.0', label='Regime I (WoE grid)', zorder=3)
ax.scatter(comp['beta'], comp['beta_eff_raw'], s=70, marker='^', c='0.7',
           edgecolor='0.0', label='Regime II (raw)', zorder=2)
ax.set_xlabel(r'logistic-WoE coefficient $\beta$')
ax.set_ylabel(r'stump effective slope $\beta_{\mathrm{eff}}$')
ax.legend(frameon=True, facecolor='white', edgecolor='0.5')
savefig(fig, '02_beta_vs_betaeff')

# ===================================================================
# 7. FIGURE C — per-feature s(x) vs WoE curves, top-6 by IV (ideal regime)
# ===================================================================
iv_df = pd.read_csv(os.path.join(TABLES, '01_information_value.csv'))
top6 = iv_df['feature'].head(6).tolist()
fig, axes = plt.subplots(2, 3, figsize=(13.5, 7.2))
for ax, c in zip(axes.ravel(), top6):
    nb = len(bin_order[c]); xx = np.arange(nb)
    ax.plot(xx, bin_woe[c], color='0.0', marker='o', lw=1.4, ms=5, label='WoE(bin)')
    ax.plot(xx, s_curves[c], color='0.45', marker='s', lw=1.4, ms=5, ls='--',
            label='s(bin) ideal')
    ax.plot(xx, s_real_bin[c], color='0.7', marker='^', lw=1.2, ms=5, ls=':',
            label='s(bin) raw')
    ax.axhline(0, color='0.1', lw=.5)
    ax.set_title(c, fontsize=9)
    ax.set_xlabel('bin (low \u2192 high; last=missing if any)'); ax.set_ylabel('log-odds contribution')
    ax.set_xticks(xx)
axes.ravel()[0].legend(frameon=True, facecolor='white', edgecolor='0.5', fontsize=8)
savefig(fig, '02_sx_vs_woe_curves')

print('\nSaved figures: 02_s_vs_woe_scatter, 02_beta_vs_betaeff, 02_sx_vs_woe_curves (png+pdf, dpi600)')
print('Saved tables : 02_logit_woe_coeffs, 02_isomorphism_ideal, 02_isomorphism_reality')

# ===================================================================
# 8. HEADLINE SUMMARY
# ===================================================================
print('\n================ ISOMORPHISM HEADLINE ================')
print('Regime I  (ideal):  mean beta_eff = %.3f | mean w-corr(s,WoE) = %.3f | mean sign-match = %.3f'
      % (iso_df['beta_eff'].mean(), iso_df['w_corr_s_woe'].mean(), iso_df['sign_match'].mean()))
print('Regime II (reality): mean beta_eff = %.3f | mean w-corr(s,WoE) = %.3f'
      % (real_df['beta_eff_raw'].mean(), real_df['w_corr_raw'].mean()))
print('Interpretation: Regime I w-corr near 1 and s(b)~WoE(b) confirms H1a/H1b under additivity;')
print('Regime II shows how far free splits drift from the WoE coordinate (isomorphism breakdown).')
print('\n02 complete.')


WoE matrix: (150000, 7) | bad rate 0.0668

=== Logistic-WoE coefficients ===
                                        beta      se   ci_lo   ci_hi
feature                                                             
RevolvingUtilizationOfUnsecuredLines  0.6800  0.0125  0.6556  0.7045
NumberOfTimes90DaysLate               0.6000  0.0111  0.5781  0.6218
NumberOfTime30-59DaysPastDueNotWorse  0.6271  0.0127  0.6021  0.6520
age                                   0.4470  0.0263  0.3956  0.4985
NumberOfOpenCreditLinesAndLoans      -0.0609  0.0428 -0.1448  0.0229
MonthlyIncome                         0.3606  0.0434  0.2755  0.4457
NumberOfDependents                    0.1895  0.0627  0.0666  0.3124


Pseudo R2 = 0.2364



=== REGIME I (ideal): stump cumulative-leaf curve vs WoE ===
                             feature  n_bins  beta_eff  w_corr_s_woe  sign_match  spearman_s_woe  max_abs_resid
RevolvingUtilizationOfUnsecuredLines       6    0.9988        1.0000      1.0000          1.0000         0.0063
             NumberOfTimes90DaysLate       3    1.0000        0.9811      1.0000          0.5000         2.8227
NumberOfTime30-59DaysPastDueNotWorse       4    1.0000        0.9844      1.0000          0.4000         2.8227
                                 age       7    1.0006        0.9985      1.0000          0.7857        12.0593
     NumberOfOpenCreditLinesAndLoans       4    0.9998        1.0000      1.0000          1.0000         0.0006
                       MonthlyIncome       6    0.9982        1.0000      0.8333          1.0000         0.0014
                  NumberOfDependents       5    1.0003        1.0000      1.0000          1.0000         0.0025



=== REGIME II (reality): raw-feature stump partial score vs WoE ===
                             feature  beta_eff_raw  w_corr_raw  n_stumps_used
RevolvingUtilizationOfUnsecuredLines        0.6798      0.9989            169
             NumberOfTimes90DaysLate        0.5505      0.9996             89
NumberOfTime30-59DaysPastDueNotWorse        0.5777      0.9994             71
                                 age        0.4391      0.9839             58
     NumberOfOpenCreditLinesAndLoans       -0.1111     -0.2927            105
                       MonthlyIncome        0.4353      0.9329             76
                  NumberOfDependents        0.1249      0.8627             32



Saved figures: 02_s_vs_woe_scatter, 02_beta_vs_betaeff, 02_sx_vs_woe_curves (png+pdf, dpi600)
Saved tables : 02_logit_woe_coeffs, 02_isomorphism_ideal, 02_isomorphism_reality

================ ISOMORPHISM HEADLINE ================
Regime I  (ideal):  mean beta_eff = 1.000 | mean w-corr(s,WoE) = 0.995 | mean sign-match = 0.976
Regime II (reality): mean beta_eff = 0.385 | mean w-corr(s,WoE) = 0.784
Interpretation: Regime I w-corr near 1 and s(b)~WoE(b) confirms H1a/H1b under additivity;
Regime II shows how far free splits drift from the WoE coordinate (isomorphism breakdown).

02 complete.
